In [67]:
import pandas as pd
import numpy as np

## 1.Ekstraksi Fitur dari Data Waktu

In [68]:

jumlah_baris = 15

# 1. Tentukan tanggal awal sebagai acuan
start_date = pd.to_datetime('2026-01-01 08:00:00')

# 2. Buat acakan "jarak hari" (0 sampai 365 hari) dan "jarak jam" (0 sampai 23 jam)
acak_hari = np.random.randint(0, 365, size=jumlah_baris)
acak_jam = np.random.randint(0, 24, size=jumlah_baris)

# 3. Tambahkan rentang acak tersebut ke tanggal awal
timestamp_acak = start_date + pd.to_timedelta(acak_hari, unit='D') + pd.to_timedelta(acak_jam, unit='h')

data = {
    'timestamp': timestamp_acak,
    'value': [i for i in range(jumlah_baris)]
}

df = pd.DataFrame(data)
df.head()

,timestamp,value
0,2026-12-05 07:00:00,0
1,2026-11-02 22:00:00,1
2,2026-12-18 04:00:00,2
3,2026-04-29 18:00:00,3
4,2026-07-06 09:00:00,4


### 1.Ekstraksi Fitur Berbasis Waktu (Hari, Bulan, Tahun)

In [69]:
df_target = df.copy()

df_target['day'] = df_target['timestamp'].dt.day
df_target['month'] = df_target['timestamp'].dt.month
df_target['year'] = df_target['timestamp'].dt.year

df_target.head()

,timestamp,value,day,month,year
0,2026-12-05 07:00:00,0,5,12,2026
1,2026-11-02 22:00:00,1,2,11,2026
2,2026-12-18 04:00:00,2,18,12,2026
3,2026-04-29 18:00:00,3,29,4,2026
4,2026-07-06 09:00:00,4,6,7,2026


### 2.Ekstraksi Fitur Tren

In [70]:
df_target = df.copy()
df_target['trend'] = df_target['value'].rolling(window=3).mean()
df_target['trend'] = df_target['trend'].fillna(0)
df_target.head()

,timestamp,value,trend
0,2026-12-05 07:00:00,0,0.0
1,2026-11-02 22:00:00,1,0.0
2,2026-12-18 04:00:00,2,1.0
3,2026-04-29 18:00:00,3,2.0
4,2026-07-06 09:00:00,4,3.0


### 3.Ekstraksi Fitur Hari Libur

In [73]:
from holidays import Indonesia

id_holidays = Indonesia()
df_target = df.copy()
df_target['is_holiday'] = df_target['timestamp'].dt.date.isin(id_holidays)
df_target.head()

,timestamp,value,is_holiday
0,2026-12-05 07:00:00,0,False
1,2026-11-02 22:00:00,1,False
2,2026-12-18 04:00:00,2,False
3,2026-04-29 18:00:00,3,False
4,2026-07-06 09:00:00,4,False


### 3.Ekstraksi Fitur Weekday vs Weekend

In [79]:
df_target = df.copy()
df_target['id_weekend'] = np.where(df_target['timestamp'].dt.weekday >=5 , 'Weekend','Weekday')
df_target.head()

,timestamp,value,id_weekend
0,2026-12-05 07:00:00,0,Weekend
1,2026-11-02 22:00:00,1,Weekday
2,2026-12-18 04:00:00,2,Weekday
3,2026-04-29 18:00:00,3,Weekday
4,2026-07-06 09:00:00,4,Weekday


## 2.Ekstraksi Fitur dari Data Tabular Lainnya

In [95]:
np.random.seed(42)
data = {
    'feature1': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16],
    'feature2': [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
    'feature3': np.random.randint(1, 100, size=15), # Fitur acak/noise
    'feature4': np.random.randint(1, 100, size=15)  # Fitur acak/noise
}
df = pd.DataFrame(data)

### 1.Interaction Terms

In [96]:
df_target = df.copy()
df_target['feature1_feature2'] = df_target['feature1']*df_target['feature2']
df_target

,feature1,feature2,feature3,feature4,feature1_feature2
0,2,5,52,2,10
1,3,6,93,88,18
2,4,7,15,30,28
3,5,8,72,38,40
4,6,9,61,2,54
5,7,10,21,64,70
6,8,11,83,60,88
7,9,12,87,21,108
8,10,13,75,33,130
9,11,14,75,76,154


### 2.Polynomial Features

In [97]:
from sklearn.preprocessing import PolynomialFeatures
df_target = df.copy()

poly = PolynomialFeatures(degree=2,include_bias=False)
poly_features = poly.fit_transform(df_target[['feature1','feature2','feature3','feature4']])
poly_features_names = poly.get_feature_names_out(['feature1','feature2','feature3','feature4'])

df_poly = pd.DataFrame(poly_features,columns=poly_features_names)
df_target = pd.concat([df,df_poly],axis=1)

df_target

,feature1,feature2,feature3,feature4,feature1,feature2,feature3,feature4,feature1^2,feature1 feature2,feature1 feature3,feature1 feature4,feature2^2,feature2 feature3,feature2 feature4,feature3^2,feature3 feature4,feature4^2
0,2,5,52,2,2.0,5.0,52.0,2.0,4.0,10.0,104.0,4.0,25.0,260.0,10.0,2704.0,104.0,4.0
1,3,6,93,88,3.0,6.0,93.0,88.0,9.0,18.0,279.0,264.0,36.0,558.0,528.0,8649.0,8184.0,7744.0
2,4,7,15,30,4.0,7.0,15.0,30.0,16.0,28.0,60.0,120.0,49.0,105.0,210.0,225.0,450.0,900.0
3,5,8,72,38,5.0,8.0,72.0,38.0,25.0,40.0,360.0,190.0,64.0,576.0,304.0,5184.0,2736.0,1444.0
4,6,9,61,2,6.0,9.0,61.0,2.0,36.0,54.0,366.0,12.0,81.0,549.0,18.0,3721.0,122.0,4.0
5,7,10,21,64,7.0,10.0,21.0,64.0,49.0,70.0,147.0,448.0,100.0,210.0,640.0,441.0,1344.0,4096.0
6,8,11,83,60,8.0,11.0,83.0,60.0,64.0,88.0,664.0,480.0,121.0,913.0,660.0,6889.0,4980.0,3600.0
7,9,12,87,21,9.0,12.0,87.0,21.0,81.0,108.0,783.0,189.0,144.0,1044.0,252.0,7569.0,1827.0,441.0
8,10,13,75,33,10.0,13.0,75.0,33.0,100.0,130.0,750.0,330.0,169.0,975.0,429.0,5625.0,2475.0,1089.0
9,11,14,75,76,11.0,14.0,75.0,76.0,121.0,154.0,825.0,836.0,196.0,1050.0,1064.0,5625.0,5700.0,5776.0


### 3.Ratio Features

In [98]:
df_target = df.copy()
df_target['feature1_to_feature2_ratio'] = df_target['feature1'] / df_target['feature2']
df_target

,feature1,feature2,feature3,feature4,feature1_to_feature2_ratio
0,2,5,52,2,0.400000
1,3,6,93,88,0.500000
2,4,7,15,30,0.571429
3,5,8,72,38,0.625000
4,6,9,61,2,0.666667
5,7,10,21,64,0.700000
6,8,11,83,60,0.727273
7,9,12,87,21,0.750000
8,10,13,75,33,0.769231
9,11,14,75,76,0.785714


## 3.Membuat Fitur Interaksi

### 1.Interaksi Berdasarkan Korelasi

In [99]:
df_target = df.copy()
df_target['corr_feature1_feature2'] = df_target['feature1'] * df_target['feature2']
correlation = df_target['corr_feature1_feature2'].corr(df_target['feature1'])
print(f"Korelasi fitur interaksi dengan target: {correlation}")

Korelasi fitur interaksi dengan target: 0.983702776615421


### 2.Interaksi Berdasarkan PCA Components

In [109]:
from sklearn.decomposition import PCA
df_target = df.copy()

pca = PCA(n_components=1)
df_target['PCA_feature'] = pca.fit_transform(df_target[['feature1','feature2','feature3','feature4']])
df_target.head()

,feature1,feature2,feature3,feature4,PCA_feature
0,2,5,52,2,8.372038
1,3,6,93,88,28.560206
2,4,7,15,30,-34.122986
3,5,8,72,38,19.307496
4,6,9,61,2,16.692482


### 3.Interaksi Berdasarkan Feature Importance 

In [108]:
from sklearn.ensemble import RandomForestRegressor
df_target = df.copy()
df_x= df_target
df_y = df_x['feature1'] * df_x['feature2']

model = RandomForestRegressor(random_state=42)
model.fit(df_x,df_y)

importance_df = pd.DataFrame({
    'fitur': df_x.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

top_2_features = importance_df['fitur'].head().tolist()
f_ke1 = top_2_features[0]
f_ke2 = top_2_features[1]

df_target[f'{f_ke1}_x_{f_ke2}'] = df_target[f_ke1] * df_target[f_ke2]

# Membuat fitur interaksi berupa pembagian (Rasio) - Tambah 1e-5 agar aman dari ZeroDivisionError
df_target[f'{f_ke1}_per_{f_ke2}'] = df_target[f_ke1] / (df_target[f_ke2] + 1e-5)

df_target.head()

,feature1,feature2,feature3,feature4,feature2_x_feature1,feature2_per_feature1
0,2,5,52,2,10,2.499988
1,3,6,93,88,18,1.999993
2,4,7,15,30,28,1.749996
3,5,8,72,38,40,1.599997
4,6,9,61,2,54,1.499998
